# Análisis de Bechdel Test + Hollywood Age Gap

En este notebook se recoge la creación de los archivos necesarios para la generación de gráficos en Flourish.

In [ ]:
## -- GRÁFICO TENDENCIA TEMPORAL --
import pandas as pd

df = pd.read_excel("movies_with_agegap.xlsx")

# Asegurar year limpio
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

# Usar clean_test (más estable que test)
df["clean_test"] = df["clean_test"].astype(str).str.strip()

# Contar películas por (year, clean_test)
counts = (
    df.groupby(["year", "clean_test"])
      .size()
      .reset_index(name="n_movies")
)

# Pasar a formato wide: una columna por categoría
area_wide = (
    counts.pivot(index="year", columns="clean_test", values="n_movies")
          .fillna(0)
          .reset_index()
)

clean_test,year,dubious,men,notalk,nowomen,ok
0,1970,0.0,0.0,0.0,0.0,1.0
1,1971,0.0,1.0,4.0,0.0,0.0
2,1972,0.0,0.0,3.0,0.0,1.0
3,1973,0.0,1.0,3.0,0.0,1.0
4,1974,0.0,1.0,4.0,0.0,2.0


In [ ]:
area_pct = area_wide.copy()
category_cols = [c for c in area_pct.columns if c != "year"]

row_sum = area_pct[category_cols].sum(axis=1)
area_pct[category_cols] = area_pct[category_cols].div(row_sum, axis=0) * 100

area_pct.head()
area_pct.to_csv("bechdel_area_pct_by_year.csv", index=False)


In [ ]:
## -- GRÁFICO DE PORCENTAJE POR GÉNERO DEL EQUIPO CREATIVO --
import pandas as pd
import numpy as np

# =========
# 1) Cargar
# =========
df = pd.read_excel("movies_with_agegap.xlsx")

# =========================================================
# 2) Asegurar booleanos robustos (0/1, True/False, NaN, etc.)
# =========================================================
for col in ["has_female_director", "has_female_writer"]:
    if col not in df.columns:
        raise KeyError(f"Falta la columna '{col}' en el Excel.")
    df[col] = df[col].fillna(0)

    # Si ya es bool, se queda; si es numérico 0/1, lo convertimos; si es otro tipo, intentamos convertir
    if df[col].dtype != bool:
        # Esto cubre int/float 0/1 típicos
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(int).astype(bool)
        else:
            # Último recurso por si vienen strings tipo "TRUE"/"FALSE"/"Yes"/"No"
            df[col] = (
                df[col].astype(str).str.strip().str.lower()
                .map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False, "y": True, "n": False})
            )
            df[col] = df[col].fillna(False).astype(bool)

# ==========================================
# 3) Seleccionar columna del test
# ==========================================
# Usamos clean_test si existe; si no, usamos test
test_col = "clean_test" if "clean_test" in df.columns else "test"
if test_col not in df.columns:
    raise KeyError("No existe 'clean_test' ni 'test' en el Excel.")

# Quitamos filas sin clasificación del test
df = df[df[test_col].notna()].copy()

# ==========================================================
# 4) Crear los 4 grupos creativos
# ==========================================================
df["creative_group"] = np.select(
    [
        (~df["has_female_director"]) & (~df["has_female_writer"]),
        ( df["has_female_director"]) & (~df["has_female_writer"]),
        (~df["has_female_director"]) & ( df["has_female_writer"]),
        ( df["has_female_director"]) & ( df["has_female_writer"]),
    ],
    [
        "Ni directora ni escritora",
        "Tiene directora",
        "Tiene escritora",
        "Directora y escritora",
    ],
    default=None  # clave: evita mezclar strings con np.nan(float)
)

df = df[df["creative_group"].notna()].copy()

# Orden de grupos para el eje X
group_order = [
    "Ni directora ni escritora",
    "Tiene escritora",
    "Tiene directora",
    "Directora y escritora",
]
df["creative_group"] = pd.Categorical(df["creative_group"], categories=group_order, ordered=True)

# Fijar orden de las categorías del test SIN cambiar nombres:
# respeta el orden si coinciden exactamente con estas etiquetas.
# Si tus categorías son otras, comenta este bloque y pandas usará orden alfabético/observado.
test_order = ["nowomen", "notalk", "men", "dubious", "ok"]
df[test_col] = pd.Categorical(df[test_col], categories=test_order, ordered=True)

# =============================================
# 5) Agregar: conteos y proporciones por grupo
# =============================================
counts = (
    df.groupby(["creative_group", test_col])
      .size()
      .reset_index(name="n")
)

counts["prop"] = counts["n"] / counts.groupby("creative_group")["n"].transform("sum")

# Dataset final para barras apiladas 100% (formato largo)
plot_df = counts.sort_values(["creative_group", test_col]).reset_index(drop=True)

# Tamaño total por grupo (útil para tooltip/nota metodológica)
group_sizes = df.groupby("creative_group").size().reset_index(name="N_total")
plot_df = plot_df.merge(group_sizes, on="creative_group", how="left")

# 1) Pivot a WIDE con proporciones (recomendado para 100% stacked)
wide_prop = (
    plot_df.pivot_table(
        index="creative_group",
        columns="clean_test",
        values="prop",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# 2) Añadir N_total como columna para tooltip o nota
N_total = plot_df[["creative_group", "N_total"]].drop_duplicates()
wide_prop = wide_prop.merge(N_total, on="creative_group", how="left")

# 3) Ordenar columnas del test sin renombrarlas
test_cols_order = ["nowomen", "notalk", "men", "dubious", "ok"]
cols = ["creative_group"] + [c for c in test_cols_order if c in wide_prop.columns] + (
    ["N_total"] if "N_total" in wide_prop.columns else []
)
wide_prop = wide_prop[cols]

# 4) Guardar para subirlo a Flourish
wide_prop.to_csv("bechdel_creativegroup_wide_prop.csv", index=False)


/tmp/ipython-input-2327815379.py:82: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["creative_group", test_col])
/tmp/ipython-input-2327815379.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts["prop"] = counts["n"] / counts.groupby("creative_group")["n"].transform("sum")
/tmp/ipython-input-2327815379.py:93: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_sizes = df.groupby("creative_group").size(

In [ ]:
## -- GRÁFICO DE EVOLUCIÓN DE GÉNERO EN EQUIPOS CREATIVOS --

import pandas as pd
import numpy as np

# =====================
# 1) Cargar dataset
# =====================
df = pd.read_excel("movies_with_agegap.xlsx")

# =====================
# 2) Columnas necesarias
# =====================
needed = ["year", "has_female_director", "has_female_writer"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas en el Excel: {missing}")

df = df[needed].copy()

# =====================
# 3) Limpieza básica
# =====================
df = df[df["year"].notna()].copy()
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df[df["year"].notna()].copy()
df["year"] = df["year"].astype(int)

# Normalizar booleanos (0/1)
for col in ["has_female_director", "has_female_writer"]:
    df[col] = df[col].fillna(0)
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(int)
    else:
        df[col] = (
            df[col].astype(str).str.strip().str.lower()
            .map({"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0, "y": 1, "n": 0})
            .fillna(0)
            .astype(int)
        )

# =====================
# 4) Crear década desde year
# =====================
df["decade"] = (df["year"] // 10) * 10   # 1994 -> 1990
df["period"] = df["decade"].astype(str) + "s"

# =====================
# 5) Crear variable "both"
# =====================
df["female_both"] = ((df["has_female_director"] == 1) & (df["has_female_writer"] == 1)).astype(int)

# =====================
# 6) Agregar por década
# =====================
agg = (
    df.groupby(["decade", "period"], as_index=False)
      .agg(
          n_movies=("year", "count"),
          pct_female_director=("has_female_director", "mean"),
          pct_female_writer=("has_female_writer", "mean"),
          pct_female_both=("female_both", "mean"),
      )
)

# Pasar a porcentaje
for col in ["pct_female_director", "pct_female_writer", "pct_female_both"]:
    agg[col] = agg[col] * 100

# Ordenar
agg = agg.sort_values("decade").reset_index(drop=True)

# =====================
# 7) Exportar CSV
# =====================
agg[[
    "period",
    "pct_female_director",
    "pct_female_writer",
    "pct_female_both",
    "n_movies"
]].to_csv("flourish_female_creative_evolution_by_decade.csv", index=False)

In [ ]:
## -- GRÁFICO DE RATINGS --
import pandas as pd

df = pd.read_excel("movies_with_agegap.xlsx")

# Columnas necesarias
needed = ["clean_test", "metascore", "imdb_rating"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas en el Excel: {missing}")

df = df[needed].copy()

# Limpieza básica
df = df[df["clean_test"].notna()].copy()
df["metascore"] = pd.to_numeric(df["metascore"], errors="coerce")
df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")

# Rango razonable (mantiene NA si está fuera)
df.loc[~df["metascore"].between(0, 100), "metascore"] = pd.NA
df.loc[~df["imdb_rating"].between(0, 10), "imdb_rating"] = pd.NA

# (Opcional) Forzar orden semántico del test en el CSV
test_order = ["nowomen", "notalk", "men", "dubious", "ok"]
df["clean_test"] = pd.Categorical(df["clean_test"], categories=test_order, ordered=True)
df = df.sort_values("clean_test").reset_index(drop=True)

# Exportar CSV
df.to_csv("flourish_violin_wide_metascore_imdb.csv", index=False)


In [ ]:
## -- GRÁFICO DE PREMIOS Y NOMINACIONES --

import pandas as pd
import re

df = pd.read_excel("movies_with_agegap.xlsx")

# Asegurar que awards es string
df["awards"] = df["awards"].astype(str)

pattern = re.compile(r"(\d+)\s+wins?\s*&\s*(\d+)\s+nominations?", re.IGNORECASE)

def extract_wins_noms(text):
    match = pattern.search(text)
    if match:
        return pd.Series({
            "wins": int(match.group(1)),
            "nominations": int(match.group(2))
        })
    else:
        return pd.Series({
            "wins": pd.NA,
            "nominations": pd.NA
        })

df[["wins", "nominations"]] = df["awards"].apply(extract_wins_noms)

awards_df = (
    df[df["wins"].notna() & df["nominations"].notna()]
    .groupby("clean_test")[["wins", "nominations"]]
    .mean()
    .reset_index()
)

awards_df.to_csv("awards.csv", index=False)


In [ ]:
## -- GRÁFICOS DE RETORNO ECONÓMICO --
import pandas as pd
import numpy as np

# =====================
# 1) Cargar dataset
# =====================
df = pd.read_excel("movies_with_agegap.xlsx")

# =====================
# 2) Columnas necesarias
# =====================
needed = [
    "title",
    "year",
    "decade_code",
    "genre",
    "clean_test",
    "budget_2013",
    "intgross_2013",
    "has_female_director",
    "has_female_writer",
]

missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas en el Excel: {missing}")

df = df[needed].copy()

# =====================
# 3) Limpieza básica
# =====================

# Eliminar filas sin test o sin datos económicos
df = df[df["clean_test"].notna()].copy()

df["budget_2013"] = pd.to_numeric(df["budget_2013"], errors="coerce")
df["intgross_2013"] = pd.to_numeric(df["intgross_2013"], errors="coerce")

# Nos quedamos solo con valores positivos
df = df[
    (df["budget_2013"] > 0) &
    (df["intgross_2013"] > 0)
].copy()

# =====================
# 4) Calcular ROI
# =====================
df["roi"] = df["intgross_2013"] / df["budget_2013"]

# Limitar ROI extremos absurdos (opcional pero MUY recomendable)
df.loc[df["roi"] > 20, "roi"] = 20

# =====================
# 5) Normalizar booleanos
# =====================
for col in ["has_female_director", "has_female_writer"]:
    df[col] = df[col].fillna(0).astype(int)

# =====================
# 6) Orden semántico del test (sin renombrar)
# =====================
test_order = ["nowomen", "notalk", "men", "dubious", "ok"]
df["clean_test"] = pd.Categorical(df["clean_test"], categories=test_order, ordered=True)

df = df.sort_values("clean_test").reset_index(drop=True)

# =====================
# 7) Exportar CSV
# =====================
df.to_csv("success_commercial_scatter.csv", index=False)